<a href="https://colab.research.google.com/github/thegurdian/ML/blob/main/frankfrut.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
"""
Diabetes Prediction - Model Comparison
Dataset: Frankfurt Hospital Diabetes Dataset (2000 patients, Germany)
Models: Logistic Regression, Decision Tree, Random Forest, Gradient Boosting,
        XGBoost, KNN, SVM
Author: Mehedi
"""

import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score,
                              f1_score, confusion_matrix, classification_report)
import xgboost as xgb

# ---------------------------------------------------------------
# 1. Load data
# ---------------------------------------------------------------
DATA_PATH = "frankfurt_diabetes_2000.csv"   # place this file next to the script
df = pd.read_csv(DATA_PATH)

# ---------------------------------------------------------------
# 2. Clean: these columns can't biologically be 0, so 0 = missing
# ---------------------------------------------------------------
cols_zero_as_missing = ['Glucose', 'BloodPressure', 'SkinThickness', 'Insulin', 'BMI']
for c in cols_zero_as_missing:
    df[c] = df[c].replace(0, np.nan)

X = df.drop(columns=['Outcome'])
y = df['Outcome']

# ---------------------------------------------------------------
# 3. Train/test split (stratified so class ratio is preserved)
# ---------------------------------------------------------------
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=42
)

# ---------------------------------------------------------------
# 4. Impute missing values (median) then scale
# ---------------------------------------------------------------
imputer = SimpleImputer(strategy='median')
X_train_imp = pd.DataFrame(imputer.fit_transform(X_train), columns=X_train.columns)
X_test_imp = pd.DataFrame(imputer.transform(X_test), columns=X_test.columns)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train_imp)
X_test_scaled = scaler.transform(X_test_imp)

# ---------------------------------------------------------------
# 5. Define models
#    (tree-based models don't need scaling, but scaled data works
#    fine for them too, so we use one consistent input for all)
# ---------------------------------------------------------------
models = {
    "Logistic Regression": LogisticRegression(max_iter=1000, random_state=42),
    "Decision Tree": DecisionTreeClassifier(
        criterion='entropy', max_depth=None, min_samples_leaf=1, random_state=42
    ),
    "Random Forest": RandomForestClassifier(
        n_estimators=400, max_depth=10, random_state=42, n_jobs=-1
    ),
    "Gradient Boosting": GradientBoostingClassifier(
        n_estimators=300, max_depth=3, random_state=42
    ),
    "XGBoost": xgb.XGBClassifier(
        n_estimators=400, max_depth=5, learning_rate=0.05,
        eval_metric='logloss', random_state=42, n_jobs=-1
    ),
    "KNN": KNeighborsClassifier(n_neighbors=21, weights='distance'),
    "SVM": SVC(kernel='rbf', C=300, gamma='scale', random_state=42),
}

# ---------------------------------------------------------------
# 6. Train, predict, evaluate
# ---------------------------------------------------------------
results = []
for name, model in models.items():
    model.fit(X_train_scaled, y_train)
    y_pred = model.predict(X_test_scaled)

    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred)
    rec = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)

    results.append({
        "Model": name,
        "Accuracy": round(acc * 100, 2),
        "Precision": round(prec * 100, 2),
        "Recall": round(rec * 100, 2),
        "F1-Score": round(f1 * 100, 2),
    })

    print(f"\n{'=' * 50}")
    print(f"{name}")
    print(f"{'=' * 50}")
    print(f"Accuracy : {acc * 100:.2f}%")
    print(f"Precision: {prec * 100:.2f}%")
    print(f"Recall   : {rec * 100:.2f}%")
    print(f"F1-Score : {f1 * 100:.2f}%")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, digits=4))

# ---------------------------------------------------------------
# 7. Summary table, sorted by accuracy
# ---------------------------------------------------------------
results_df = pd.DataFrame(results).sort_values(by="Accuracy", ascending=False)
results_df.to_csv("model_comparison_results.csv", index=False)

print("\n\n" + "=" * 60)
print("SUMMARY - ALL MODELS")
print("=" * 60)
print(results_df.to_string(index=False))


Logistic Regression
Accuracy : 79.75%
Precision: 75.45%
Recall   : 60.58%
F1-Score : 67.21%
Confusion Matrix:
[[236  27]
 [ 54  83]]
              precision    recall  f1-score   support

           0     0.8138    0.8973    0.8535       263
           1     0.7545    0.6058    0.6721       137

    accuracy                         0.7975       400
   macro avg     0.7842    0.7516    0.7628       400
weighted avg     0.7935    0.7975    0.7914       400


Decision Tree
Accuracy : 97.25%
Precision: 94.37%
Recall   : 97.81%
F1-Score : 96.06%
Confusion Matrix:
[[255   8]
 [  3 134]]
              precision    recall  f1-score   support

           0     0.9884    0.9696    0.9789       263
           1     0.9437    0.9781    0.9606       137

    accuracy                         0.9725       400
   macro avg     0.9660    0.9738    0.9697       400
weighted avg     0.9731    0.9725    0.9726       400


Random Forest
Accuracy : 96.75%
Precision: 95.59%
Recall   : 94.89%
F1-Score : 95.2